In [ ]:
!nvidia-smi

In [ ]:
!git clone https://@github.com/rayzhao27/bert4rec.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch torchvision numpy pandas scikit-learn scipy requests tqdm tensorboard wandb joblib fastapi uvicorn httpx pydantic

In [ ]:
!git -C /content/bert4rec pull

In [ ]:
!python /content/bert4rec/data/download.py --data_dir /content/bert4rec/data

In [ ]:
!python /content/bert4rec/data/preprocess.py --data_dir /content/bert4rec/data --min_rating 0

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

Baseline training:

In [ ]:
!python /content/bert4rec/training/trainer.py \
  --data_dir /content/bert4rec/data \
  --hidden_size 256 \
  --num_hidden_layers 4 \
  --num_attention_heads 4 \
  --intermediate_size 1024 \
  --hidden_dropout_prob 0.2 \
  --attention_probs_dropout 0.2 \
  --learning_rate 1e-3 \
  --warmup_steps 100 \
  --weight_decay 0.01 \
  --epochs 300 \
  --num_workers 2 \
  --checkpoint_dir /content/drive/MyDrive/bert4rec_checkpoints/baseline \
  --log_dir /content/drive/MyDrive/bert4rec_runs/baseline

Multi-feature training:

In [ ]:
!python /content/bert4rec/training/trainer.py \
  --data_dir /content/bert4rec/data \
  --use_features \
  --hidden_size 256 \
  --num_hidden_layers 4 \
  --num_attention_heads 4 \
  --intermediate_size 1024 \
  --hidden_dropout_prob 0.2 \
  --attention_probs_dropout 0.2 \
  --learning_rate 1e-3 \
  --warmup_steps 100 \
  --weight_decay 0.01 \
  --epochs 300 \
  --num_workers 2 \
  --checkpoint_dir /content/drive/MyDrive/bert4rec_checkpoints/multifeature \
  --log_dir /content/drive/MyDrive/bert4rec_runs/multifeature

Baseline evaluate:

In [ ]:
!python /content/bert4rec/evaluation/evaluator.py \
  --checkpoint /content/drive/MyDrive/bert4rec_checkpoints/baseline/best_model.pt \
  --data_dir /content/bert4rec/data

Multi-feature evaluate:

In [ ]:
!python /content/bert4rec/evaluation/evaluator.py \
  --checkpoint /content/drive/MyDrive/bert4rec_checkpoints/multifeature/best_model.pt \
  --data_dir /content/bert4rec/data

In [ ]:
import torch
for name in ["baseline", "multifeature"]:
    ckpt = torch.load(f'/content/drive/MyDrive/bert4rec_checkpoints/{name}/best_model.pt', map_location='cpu')
    print(f"── {name} ──")
    print("  epoch:", ckpt['epoch'])
    print("  val_loss:", ckpt['best_loss'])
    print("  use_features:", ckpt['cfg'].get('use_features', False))
    print("  num_genres:", ckpt['cfg'].get('num_genres', 0))
    print()